# Document Similarity Search Using LSA

Build a lightweight semantic search engine over technical documents.

**Portfolio category:** NLP similarity

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Curated, privacy-safe text corpus

In [ ]:
documents = {
    "Lakehouse": "databricks lakehouse delta tables governance batch streaming",
    "Feature Store": "machine learning features reuse training serving consistency",
    "RAG Evaluation": "retrieval relevance faithfulness context precision evaluation",
    "Power BI": "semantic model dax dashboards visual analytics business",
    "Time Series": "forecasting seasonality trend lag rolling validation",
    "MLOps": "model registry deployment monitoring drift ci cd automation",
    "Vector Search": "embeddings nearest neighbour semantic retrieval index",
    "Data Quality": "schema validation completeness duplicates lineage observability",
    "Experimentation": "ab testing hypothesis power sample significance metrics",
    "Clustering": "unsupervised segmentation similarity silhouette profiles",
    "Anomaly Detection": "outlier isolation forest rare events ranking alerts",
    "SQL Analytics": "joins windows aggregations query optimisation warehouse",
}
source_names, source_texts = list(documents), list(documents.values())
target_names, target_texts = source_names, source_texts
query_label = "document"

## 3. Corpus quality

In [ ]:
quality = pd.DataFrame({
    "name": source_names + target_names,
    "text": source_texts + target_texts,
})
quality["tokens"] = quality["text"].str.split().str.len()
display(quality)

## 4. TF-IDF representation

In [ ]:
all_texts = source_texts + target_texts
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
X = vectorizer.fit_transform(all_texts)
n_components = min(8, X.shape[0] - 1, X.shape[1] - 1)
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
latent = Normalizer().fit_transform(svd.fit_transform(X))
source_vectors = latent[: len(source_texts)]
target_vectors = latent[len(source_texts):]
similarity = cosine_similarity(source_vectors, target_vectors)

## 5. Rank semantic matches

In [ ]:
rows = []
for source_index, source_name in enumerate(source_names):
    order = np.argsort(similarity[source_index])[::-1]
    for rank, target_index in enumerate(order[:3], start=1):
        rows.append({
            query_label: source_name,
            "rank": rank,
            "match": target_names[target_index],
            "score": similarity[source_index, target_index],
        })
ranking = pd.DataFrame(rows)
display(ranking.round(3))

## 6. Ranking diagnostics

In [ ]:
top_scores = ranking.query("rank == 1")["score"].to_numpy()
second_scores = ranking.query("rank == 2")["score"].to_numpy()
display(pd.Series({
    "explained_variance": svd.explained_variance_ratio_.sum(),
    "mean_top_score": top_scores.mean(),
    "mean_top_vs_second_margin": (top_scores - second_scores).mean(),
}).to_frame("value"))

## 7. Similarity heatmap

In [ ]:
sns.heatmap(
    pd.DataFrame(similarity, index=source_names, columns=target_names),
    cmap="Blues",
    annot=True,
    fmt=".2f",
)
plt.title("Latent semantic similarity")
plt.tight_layout()

## 8. Explain a match with overlapping terms

In [ ]:
source_index = 0
target_index = int(np.argmax(similarity[source_index]))
source_terms = set(source_texts[source_index].split())
target_terms = set(target_texts[target_index].split())
display(pd.Series({
    "query": source_names[source_index],
    "best_match": target_names[target_index],
    "shared_terms": ", ".join(sorted(source_terms & target_terms)),
}).to_frame("value"))

## 9. Key findings

Similarity scores support ranking, not hiring or access decisions. Human review and bias checks remain essential.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For document similarity search using lsa,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.